In [1]:
import pandas as pd

In [3]:
item_level = pd.read_parquet("data/processed/item_level.parquet")

item_df = item_level[
    (item_level["order_status"] == "delivered") & (~item_level["is_invalid_timestamps"])
].copy()

#### Seller SLA compliance: did the seller hand off to carrier on time?

In [4]:
item_df["seller_delay"] = (
    item_df["order_delivered_carrier_date"] - item_df["shipping_limit_date"]
).dt.days
item_df["seller_missed_sla"] = item_df["seller_delay"] > 0

print(item_df["seller_delay"].describe())
print(f"\nSeller SLA miss rate: {item_df['seller_missed_sla'].mean() * 100:.2f}%")

count    108603.000000
mean         -3.812593
std           5.841008
min       -1047.000000
25%          -6.000000
50%          -4.000000
75%          -2.000000
max         116.000000
Name: seller_delay, dtype: float64

Seller SLA miss rate: 4.72%


#### Split total delivery delay into seller-side vs carrier-side portions
* seller_stage: purchase -> carrier handoff (vs shipping_limit_date)
* carrier_stage: carrier handoff -> customer delivery (vs estimated_delivery_date)

In [5]:
item_df["carrier_transit_days"] = (
    item_df["order_delivered_customer_date"] - item_df["order_delivered_carrier_date"]
).dt.days

# Estimated transit budget: time carrier "should" have had, based on ETA minus seller's allotted time
item_df["estimated_transit_days"] = (
    item_df["order_estimated_delivery_date"] - item_df["shipping_limit_date"]
).dt.days
item_df["carrier_delay"] = item_df["carrier_transit_days"] - item_df["estimated_transit_days"]

# Attribute overall lateness to whichever stage overshot its own budget
item_df["is_late_overall"] = (
    item_df["order_delivered_customer_date"] > item_df["order_estimated_delivery_date"]
)

late_items = item_df[item_df["is_late_overall"]].copy()
late_items["primary_cause"] = late_items.apply(
    lambda row: "seller"
    if row["seller_missed_sla"] and not (row["carrier_delay"] > 0)
    else ("carrier" if row["carrier_delay"] > 0 and not row["seller_missed_sla"] else "both"),
    axis=1,
)

print(late_items["primary_cause"].value_counts(normalize=True) * 100)

primary_cause
carrier    78.442217
seller     12.201867
both        9.355917
Name: proportion, dtype: float64


#### Overall attribution of lateness
- Carrier-caused: 78.4%
- Seller-caused: 12.2%
- Both stages overshot: 9.4%

#### Takeaway
Most delay is carrier-side, not seller-side. Seller SLA miss rate overall is only 4.72% (median seller_delay = -4 days, i.e. sellers usually ship early). This shifts priority — carrier/logistics improvements matter more platform-wide than seller enforcement.

#### Update risk-seller list with seller_missed_sla_rate

In [6]:
risk_sellers = pd.read_parquet("data/processed/risk_sellers.parquet")

seller_sla = (
    item_df.drop_duplicates(subset=["order_id", "seller_id"])
    .groupby("seller_id")
    .agg(
        seller_missed_sla_rate=("seller_missed_sla", "mean"),
        avg_seller_delay=("seller_delay", "mean"),
        n_orders=("order_id", "nunique"),
    )
    .query("n_orders >= 30")
)

risk_sellers_updated = risk_sellers.merge(
    seller_sla, left_index=True, right_index=True, how="left", suffixes=("", "_sla")
)
print(risk_sellers_updated.sort_values("seller_missed_sla_rate", ascending=False))

                                  total_revenue  n_orders  avg_review  \
seller_id                                                               
54965bbe3e4f07ae045b90b0b8541f52       10169.90        71    3.013514   
88460e8ebdecbfecb5f9601833981930       31080.70       243    3.351351   
a7f13822ceb966b076af67121f87b063       12017.69        73    3.493976   
c60b801f2d52c7f7f91de00870882a75       11296.50        39    3.435897   
2eb70248d66e0e3ef83659f71b244378       38990.72       187    2.809278   
1ca7077d890b907f89be8c954a02686a       12474.64       108    2.269841   
8444e55c1f13cd5c179851e5ca5ebd00       21480.96        92    3.156863   
8e6d7754bc7e0f22c96d255ebda59eba       14107.38        84    2.992248   
710e3548e02bc1d2831dfc4f1b5b14d4       21939.27       129    3.455224   
712e6ed8aa4aa1fa65dab41fed5737e4       39385.00        77    3.464286   
7c67e1448b00f6e969d365cea6b010ab      186400.11       972    3.351412   
5058e8c1e82653974541e83690655b4a        9812.83    

#### At the individual seller level
This flips for the risk-seller list:
- `54965bbe3e4f07ae045b90b0b8541f52`: 43.7% SLA miss rate, avg 8.5 days late handoff — clearly a **seller-side** problem, not carrier
- `88460e8ebdecbfecb5f9601833981930`, `a7f13822...`, `c60b801f2...`: all 30–40% SLA miss rate — same pattern
- `7c67e1448b...` (the largest risk seller by revenue): only 14.4% SLA miss rate, avg seller_delay is *negative* (-2.5 days, ships early) — its problem is **not fulfillment speed**, so the root cause of its low review score must lie elsewhere (product quality, carrier assigned to it, etc.)
- 4 sellers (`40db9e9a...`, `b1b39487...`, `821fb029...`, `f08a5b9d...`) have NaN — too few qualifying orders (<30) to compute reliably, should be excluded from SLA-based conclusions

#### Seller SLA miss rate by state — is Northeast a seller problem or a carrier problem?

In [7]:
sla_by_state = (
    item_df.drop_duplicates(subset=["order_id", "seller_id"])
    .groupby("customer_state")
    .agg(
        seller_missed_sla_rate=("seller_missed_sla", "mean"),
        avg_carrier_delay=("carrier_delay", "mean"),
        n_orders=("order_id", "nunique"),
    )
    .query("n_orders >= 30")
    .sort_values("seller_missed_sla_rate", ascending=False)
)
sla_by_state.head(10)

,seller_missed_sla_rate,avg_carrier_delay,n_orders
customer_state,,,
AL,0.060759,-4.997468,393
SE,0.059524,-6.053571,333
RN,0.059448,-9.745223,467
PB,0.052632,-9.436647,509
MA,0.050209,-5.412831,705
ES,0.050125,-6.547870,1968
RR,0.050000,-13.975000,40
CE,0.049606,-6.937008,1260
SC,0.048698,-7.463760,3492


#### By state
Seller SLA miss rate is fairly uniform (~5–6%) across top states — no state stands out as a "seller prep" bottleneck. Combined with `avg_carrier_delay` being consistently negative (carriers usually beat their own budget), Northeast Brazil's lateness problem looks driven by **absolute transit distance/time**, not by sellers in that region shipping late or carriers underperforming relative to their allotted budget.

#### Key Insight
The earlier assumption that "risk sellers" are uniformly at fault needs refinement:
- Some risk sellers (`54965bbe...`, `88460e8e...`, `a7f13822...`) have a genuine **fulfillment problem** — actionable via seller SLA enforcement/reminders
- The biggest-revenue risk seller (`7c67e1448b...`) does **not** — its low review score needs a different root cause (product/category issue, or carrier assigned to its shipments)
- Platform-wide, delay is overwhelmingly a **carrier/transit issue** (78%), reinforcing the earlier ETA-calibration finding: the fix is more about setting honest expectations and carrier capacity in slow regions, not chasing seller behavior